# 챗봇 - 검색기반

## 실행시 주의사항
### 1. 'ChatbotData.csv'
업로드 확인
### 2. 'cached_emb_data.npy'
업로드 확인
* 없으면 새로 임배딩 시작 => 느림

In [144]:
# !pip install sentence-transformers

In [145]:
# 임포트
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from sentence_transformers import SentenceTransformer
import pandas as pd
import os

In [146]:
# 데이터로드
df = pd.read_csv('ChatbotData.csv')
df

,Q,A,label
0,12시 땡!,하루가 또 가네요.,0
1,1지망 학교 떨어졌어,위로해 드립니다.,0
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.,0
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠.,0
4,PPL 심하네,눈살이 찌푸려지죠.,0
...,...,...,...
11818,훔쳐보는 것도 눈치 보임.,티가 나니까 눈치가 보이는 거죠!,2
11819,훔쳐보는 것도 눈치 보임.,훔쳐보는 거 티나나봐요.,2
11820,흑기사 해주는 짝남.,설렜겠어요.,2
11821,힘든 연애 좋은 연애라는게 무슨 차이일까?,잘 헤어질 수 있는 사이 여부인 거 같아요.,2


In [147]:
# 모델 불러오기 : Sentence-Bert - 수많은 한국어 텍스트를 학습한 모델
model = SentenceTransformer('jhgan/ko-sroberta-multitask')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: jhgan/ko-sroberta-multitask
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [148]:
# 코사인 유사도를 활용하기 위해
# 질문 데이터 임베딩
def load_emb_data(file_path):
  if os.path.exists(file_path):
    emb_data = np.load(file_path)
  else:
    print(f'{file_path} 파일이 없습니다.')
    print('새로 임베딩을 시작합니다.')
    print('질문 데이터 변환 중...')
    emb_data = model.encode(df['Q'])
    print('변환 완료!')
  return emb_data
emb_data = load_emb_data('cached_emb_data.npy')

In [149]:
# 함수 : 챗봇이 생성한 답변
def get_chatbot_response(user_input):
  # 코사인 유사도를 비교하기 위해 사용자 질문을 임베딩
  user_emb = model.encode([user_input])

  # 코사인 유사도를 계산
  cos_sime = cosine_similarity(user_emb, emb_data)

  # 가장 높은 유사도를 가진 인덱스(위치) 찾기
  best_idx = np.argmax(cos_sime)
  # 해당 인덱스에 있는 답변 반환
  # 선택한 답변
  answer = df.iloc[best_idx]['A']
  # 유사도
  score = cos_sime[0][best_idx]

  return answer, score

In [150]:
while True:
  text = input('대화 : ')

  # 종료 조건
  if text == '종료': break

  respone, score = get_chatbot_response(text)
  print(f'답변 : {respone}({score*100:.1f}%)')

대화 : 종료


In [151]:
np.save('cached_emb_data.npy', emb_data)

# 챗봇 - 생성 기반

In [152]:
# !pip install transformers torch

In [153]:
import torch
from transformers import PreTrainedTokenizerFast, GPT2LMHeadModel
import re

In [154]:
# 모델 불러오기(kogpt2)
tokenizer = PreTrainedTokenizerFast.from_pretrained(
    "skt/kogpt2-base-v2",
    # bos_token : 문장의 시작을 알리는 기호
    # eos_token : 문장의 끝을 알리는 기호
    # unk_token : 알수없는 단어를 처리하는 기호
    # pad_token : 문장의 길이를 맞추기 위해 넣는 기호
    # mask_token : 문장의 특정 단어를 가리고 맞히는 학습할 때 사용하는 기호
    bos_token='</s>', eos_token='</s>', unk_token='<unk>',
    pad_token='<pad>', mask_token='<mask>'
)
model = GPT2LMHeadModel.from_pretrained("skt/kogpt2-base-v2")

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 
transformer.h.{0...11}.attn.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [155]:
# 문장을 생성하는 함수
def generate_answer(prompt, max_len=50):

  # 입력 문장을 토큰화
  input_ids = tokenizer.encode(prompt, return_tensors="pt")

  # 이어질 문장을 생성
  output = model.generate(
      input_ids,
      max_length=max_len, # 생성할 문장의 최대 길이
      repetition_penalty=2.0, # 똑같은 말을 하면 벌칙
      do_sample=True, # False이면 가장 확률이 높은 단어를 고름 => 답변이 항상 같음
      top_k=50, # 단어 후보 중 상위 50개 안에서 고름
      top_p=0.95, # 후보 단어들의 누적 확률이 95% 되는 지점까지만 후보군에 포함
      pad_token_id=tokenizer.pad_token_id
  )
  # 생성된 토큰들을 한글 문장으로 복원
  return tokenizer.decode(output[0], skip_special_tokens=True)

In [156]:
# 생성한 답변 문장의 끝을 깔끔하게 정리하기 위한 함수
def clean_end_stop_generation(text, max_len=1000):
  # 답변을 .?!를 기준으로 나눔(한 문단을 문장으로 나눔)
  sentences = re.split(r'([.?!])', text)
  clean_text = ""
  # 아래 반복문에서 len(sentences) - 1 , -1를 하는 이유를 설명
  # => 문장의 끝(.?!)이 없는 문장을 버리기 위해
  # 문장 + 문장끝(.?!)를 붙이기 위해
  # 문장1, 문장2, 문장3(마무리 안됨)
  # [문장1, . . 문장2, . 문장3]
  # 0       1   2      3  4
  # 0 => 0 + 1,
  # 2 => 2 + 3
  # 4
  for i in range(0, len(sentences) - 1, 2):
    clean_text += sentences[i] + sentences[i+1]
    if len(clean_text) > max_len:
      break
  return clean_text if clean_text else text.strip()

In [157]:
def test_generate(text):
  answer = generate_answer(text)
  cleaned_answer = clean_end_stop_generation(answer, 50)
  print('-'*30)
  print(f"{cleaned_answer}")
  print('-'*30)

In [158]:
test_generate("오늘 비가 와요")
test_generate("공부하기 싫어요")
test_generate("곧 점심 시간이에요")

------------------------------
오늘 비가 와요라고 하셨는데요.
네. 오호 태풍 찬홈의 영향권 안에 있었던 제주도는 호우경보가 내려져 있고 서울 등 중부지방은 약하게 눈이 내리고 있습니다.
------------------------------
------------------------------
공부하기 싫어요?"
나는 그렇게 생각해서라도 그 여자에게 '그래도 네가 싫었으면 어쩔 수 없이 너한테 잘 보이기 위해 노력하겠지' 하고 말해야 했을 뿐더러 그녀가 무슨 잘못을 했는지 알지 못하게 되었다.
------------------------------
------------------------------
곧 점심 시간이에요~
이벤트 신청만 하면 돼서 기분 좋게 다녀왔어요
진짜 맛있게 먹었으니 정말 좋았어요.
------------------------------


# 챗봇-RAG 기반

In [168]:
# !pip install sentence-transformers

In [169]:
# 임포트

In [170]:
import pandas as pd
# 데이터 불러오기
df = pd.read_csv('ChatbotData.csv')

In [171]:
import torch
from transformers import PreTrainedTokenizerFast, GPT2LMHeadModel
import re
# 검색 모델
ret_model = SentenceTransformer('jhgan/ko-sroberta-multitask')
# 생성 모델
tokenizer = PreTrainedTokenizerFast.from_pretrained(
    "skt/kogpt2-base-v2",
    # bos_token : 문장의 시작을 알리는 기호
    # eos_token : 문장의 끝을 알리는 기호
    # unk_token : 알수없는 단어를 처리하는 기호
    # pad_token : 문장의 길이를 맞추기 위해 넣는 기호
    # mask_token : 문장의 특정 단어를 가리고 맞히는 학습할 때 사용하는 기호
    bos_token='</s>', eos_token='</s>', unk_token='<unk>',
    pad_token='<pad>', mask_token='<mask>'
)
gen_model = GPT2LMHeadModel.from_pretrained("skt/kogpt2-base-v2")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: jhgan/ko-sroberta-multitask
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 
transformer.h.{0...11}.attn.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [172]:
# 질문 데이터 임베딩
def load_emb_data(file_path):
  if os.path.exists(file_path):
    emb_data = np.load(file_path)
  else:
    print(f'{file_path} 파일이 없습니다.')
    print('새로 임베딩을 시작합니다.')
    print('질문 데이터 변환 중...')
    emb_data = model.encode(df['Q'])
    print('변환 완료!')
  return emb_data
emb_data = load_emb_data('cached_emb_data.npy')

In [187]:
def get_rag_chatbot_response(user_input, max_len=50):
  # 1. 검색기반으로 모범 답안 선택
  # 코사인 유사도를 비교하기 위해 사용자 질문을 임베당
  user_emb = ret_model.encode([user_input])

  # 코사인 유사도를
  cos_sims = cosine_similarity(user_emb, emb_data)

  # 가장 높은 유사도를 가진 인뎃스(위치) 찾기
  best_idx = np.argmax(cos_sims)
  # 해당 인덱스에 있는 답변
  # 선택된 답변
  ret_answer = df.iloc[best_idx]['A']
  # 2. 생성기반으로 모범 답안을 기준으로 이어서 답안 생성
  prompt = f'질문: {user_input}\n정보:{ret_answer}\n답변:{ret_answer}.'

  # 입력 문장을 토큰화
  input_ids = tokenizer.encode(prompt, return_tensors="pt")

  # 이어질 문장을 생성
  output = gen_model.generate(
      input_ids,
      max_length=max_len,# 생성할 문장의 최대 길이
      repetition_penalty=2.0,# 똑같은 말을 하면 벌칙
      do_sample=True, # False이면 가장 확률이 높은 단어를 고름 => 답변이 항상 같음
      top_k=50, # 단어 후보 중 상위 50개 안에서 고름
      top_p=0.95, # 후보 단어들의 누적 확률이 95% 되는 지점까지만 후보군에 포함
      pad_token_id=tokenizer.pad_token_id # 문장의 길이들 맞추기위해
  )
  # 생성된 토큰들을 한글 문장으로 복원
  decoded =  tokenizer.decode(output[0], skip_special_tokens=True)
  return clean_end_stop_generation(decoded, max_len)

In [188]:
# 생성한 답변 문장의 끝을 깔끔하게 정리하기 위한 함수
def clean_end_stop_generation(text, max_len=50):
  if "답변:" in text:
    # 답변:을 기준으로 분리 후 제일 마지막에 있는 내용을 가져옴
    # 질문:aaa\n정보:bbb\n답변:ccc
    text = text.split("답변:")[-1].strip()

  sentences = re.split(r'([.?!])', text)
  clean_text = ""

  for i in range(0, len(sentences) - 1, 2):
    clean_text += sentences[i] + sentences[i+1]
    if len(clean_text) > max_len:
      break
  return clean_text if clean_text else text.strip()

In [189]:
def test_rag_generate(text):
  cleaned_answer = get_rag_chatbot_response(text, 50)
  print('-'*30)
  print(f"{cleaned_answer}")
  print('-'*30)

In [190]:
test_rag_generate('점심을 먹어서 배부르다')
test_rag_generate('비가 오네요')
test_rag_generate('썸타고 있어요')
test_rag_generate('세상에서 젤 아픈 이야기는 뭐야')

------------------------------
즐거운 시간 보내시길 바랍니다...
아침에 퇴근하시는 분들은 오늘도 출근시간대로 늦으시고요.
------------------------------
------------------------------
그리고 멈출 거예요...
정말 너무 많이 퍼주신 것 같애요.
결론적으로 좀 더 나아졌으면 좋겠어요.
------------------------------
------------------------------
연애로 이어지길 바랄게요...
------------------------------
------------------------------
이뤄지지 못한 모든 사랑이야기죠...
------------------------------
